In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd


URL = "https://vahan.parivahan.gov.in/vahan4dashboard/vahan/dashboardview.xhtml"

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

driver.get(URL)

wait = WebDriverWait(driver, 20)

years = ["2022","2023","2024","2025","2026"]
months = ["JAN","FEB","MAR","APR","MAY","JUN",
          "JUL","AUG","SEP","OCT","NOV","DEC"]

all_data = []

from selenium.webdriver.common.action_chains import ActionChains

for year in years:
    print(f"\nProcessing Year: {year}")

    year_link = wait.until(
        EC.element_to_be_clickable((By.XPATH, f"//a[normalize-space()='{year}:']"))
    )
    driver.execute_script("arguments[0].click();", year_link)

    # Wait for Vehicle Class panel for that year
    wait.until(
        EC.presence_of_element_located(
            (By.XPATH, f"//div[contains(text(),'Vehicle Class({year})')]")
        )
    )

    for month in months:
        print(f"  Scraping {month}-{year}")

        # Find month button ONLY inside month row
        month_link = wait.until(
            EC.element_to_be_clickable((
                By.XPATH,
                f"//div[contains(@class,'resp-month')]//a[normalize-space()='{month}']"
            ))
        )

        driver.execute_script("arguments[0].click();", month_link)

        # Wait for table header to change to correct month
        wait.until(
            EC.presence_of_element_located(
                (By.XPATH, f"//div[contains(text(),'Vehicle Class({month}-{year[-2:]})')]")
            )
        )

        # Now extract table safely
        table = driver.find_element(
            By.XPATH,
            f"//div[contains(text(),'Vehicle Class({month}-{year[-2:]})')]/ancestor::div[contains(@class,'ui-panel')]//table"
        )

        rows = table.find_elements(By.TAG_NAME, "tr")

        for row in rows[1:]:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 2:
                vehicle_class = cols[0].text
                count = cols[1].text
                all_data.append([year, month, vehicle_class, count])

# Save
df = pd.DataFrame(all_data, columns=["Year","Month","Vehicle Class","Count"])
df.to_csv("vahan_vehicle_class_monthly.csv", index=False)

print("\nScraping Complete!")
driver.quit()


ModuleNotFoundError: No module named 'undetected_chromedriver'

In [17]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

BASE_URL = "https://vahan.parivahan.gov.in/vahan4dashboard/vahan/view/reportview.xhtml"

session = requests.Session()

# Step 1: Initial GET to fetch ViewState
response = session.get(BASE_URL)
soup = BeautifulSoup(response.text, "html.parser")

viewstate = soup.find("input", {"name": "javax.faces.ViewState"})["value"]

print("ViewState captured ✅")

years = ["2022"]
months = ["JAN", "FEB", "MAR"]

all_data = []

for year in years:
    for month in months:
        print(f"Scraping {month}-{year}")

        payload = {
            "javax.faces.partial.ajax": "true",
            "javax.faces.source": "j_idt38",
            "javax.faces.partial.execute": "@all",
            "javax.faces.partial.render": "j_idt38",
            "j_idt38": "j_idt38",
            "year": year,
            "month": month,
            "javax.faces.ViewState": viewstate
        }

        headers = {
            "Faces-Request": "partial/ajax",
            "User-Agent": "Mozilla/5.0"
        }

        r = session.post(BASE_URL, data=payload, headers=headers)

        soup = BeautifulSoup(r.text, "html.parser")
        table = soup.find("table")

        if table:
            df = pd.read_html(StringIO(str(table)))[0]
            df["Year"] = year
            df["Month"] = month
            all_data.append(df)
            print("   ✓ Done")
        else:
            print("   ✗ No table")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df.to_csv("vahan_vehicle_data.csv", index=False)
    print("\n✅ Data saved.")
else:
    print("\n❌ No data scraped.")


ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))